# two-optimizers-alternating-step — worked example 3: n_critic D-steps per G-step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `two-optimizers-alternating-step`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

WGAN trains the critic harder than the generator: each outer iteration runs `n_critic` discriminator updates (each with fresh noise) followed by a single generator update. Sampling new `z` per inner step keeps the critic from overfitting to one noise batch.

## Worked solution

We run one WGAN outer iteration with several critic updates.

1. Inner loop: for each of `n_critic` steps we zero `D_opt`, sample fresh `z`, detach the fake, compute the simplified Wasserstein loss, backprop, step `D_opt`, and record the loss.
2. Outer: a single generator update — fresh `z`, no detach, `-D(fake).mean()`, backprop, `G_opt.step()`.
3. The critic therefore takes `n_critic` steps for every one generator step.

We print the list of critic losses (length `n_critic`) and the single generator loss.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)
G = nn.Linear(4, 6)
D = nn.Linear(6, 1)
G_opt = t.optim.SGD(G.parameters(), lr=0.01)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)

def wgan_iter(G, D, G_opt, D_opt, x_real, z_dim, n_critic):
    B = x_real.shape[0]
    d_losses = []
    for _ in range(n_critic):
        D_opt.zero_grad()
        z = t.randn(B, z_dim)
        fake = G(z).detach()
        loss_D = (D(fake) - D(x_real)).mean()
        loss_D.backward(); D_opt.step()
        d_losses.append(loss_D.item())
    G_opt.zero_grad()
    z = t.randn(B, z_dim)
    loss_G = -D(G(z)).mean()
    loss_G.backward(); G_opt.step()
    return d_losses, loss_G.item()

x_real = t.randn(8, 6)
d_losses, lG = wgan_iter(G, D, G_opt, D_opt, x_real, z_dim=4, n_critic=3)
print('num critic steps:', len(d_losses))
print('loss_G:', round(lG, 4))